In [2]:
! pip install beautifulsoup4
! pip install requests


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: C:\Users\user\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: C:\Users\user\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [3]:
import requests
import time
from bs4 import BeautifulSoup

ModuleNotFoundError: No module named 'requests'

In [ ]:
CHANNEL = "telegram"
URL = f"https://t.me/s/{CHANNEL}"
HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

seen_ids = set()

def fetch_messages():
    res = requests.get(URL, headers=HEADERS, timeout=10)
    soup = BeautifulSoup(res.text, "html.parser")

    messages = soup.select("div.tgme_widget_message")

    new_texts = []

    for msg in messages:
        msg_id = msg.get("data-post")
        if msg_id in seen_ids:
            continue

        seen_ids.add(msg_id)

        text_div = msg.select_one(".tgme_widget_message_text")
        if text_div:
            text = text_div.get_text(separator=" ", strip=True)
            new_texts.append(text)

    return new_texts


while True:
    try:
        new_msgs = fetch_messages()
        if new_msgs:
            with open(f"{CHANNEL}.txt", "a", encoding="utf-8") as f:
                for t in new_msgs:
                    f.write(t + "\n")

            print(f"[+] {len(new_msgs)} new messages")

    except Exception as e:
        print("error:", e)

    time.sleep(3)  # 3초 간격 스트리밍


In [ ]:
CHANNELS = ["telegram", "durov", "cointrendz"]
HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

seen_ids = {ch: set() for ch in CHANNELS}

def fetch_channel(channel):
    url = f"https://t.me/s/{channel}"
    res = requests.get(url, headers=HEADERS, timeout=10)
    soup = BeautifulSoup(res.text, "html.parser")

    msgs = soup.select("div.tgme_widget_message")
    new = []

    for msg in msgs:
        msg_id = msg.get("data-post")
        if msg_id in seen_ids[channel]:
            continue

        seen_ids[channel].add(msg_id)

        text_div = msg.select_one(".tgme_widget_message_text")
        if text_div:
            new.append(text_div.get_text(" ", strip=True))

    return new


while True:
    for ch in CHANNELS:
        try:
            texts = fetch_channel(ch)
            if texts:
                with open(f"{ch}.txt", "a", encoding="utf-8") as f:
                    for t in texts:
                        f.write(t + "\n")

                print(f"[{ch}] +{len(texts)}")

        except Exception as e:
            print(ch, "error", e)

    time.sleep(3)